In [ ]:
import openpyxl
import numpy as np
import calendar
from datetime import date, datetime

DATA_DIR = '/workspace/data/'
XLSX = DATA_DIR + 'MO15-More-Money-Please-Data.xlsx'

wb = openpyxl.load_workbook(XLSX, data_only=True)

# Extract time-series data from sheets
# Columns C12=2014, C13=2015, ..., C34=2036

def get_row_data(ws, row, start_col=13, end_col=35):
    """Extract year->value mapping from a row."""
    data = {}
    for c in range(start_col, end_col):
        yr_cell = ws.cell(7, c).value  # Row 7 has period end dates
        val = ws.cell(row, c).value
        if yr_cell and val is not None and isinstance(val, (int, float)):
            yr = yr_cell.year if isinstance(yr_cell, datetime) else int(yr_cell)
            data[yr] = val
    return data

def get_row_data_with_2014(ws, row):
    """Include column 12 (2014) as well."""
    data = get_row_data(ws, row)
    v12 = ws.cell(row, 12).value
    if v12 is not None and isinstance(v12, (int, float)):
        data[2014] = v12
    return data

fs = wb['Financial Statements']
wk = wb['Workings']

# Project Cashflow (FS row 19)
project_cf = get_row_data(fs, 19)
# Senior debt repayments (FS row 21)
sr_repay = get_row_data(fs, 21)
# Senior debt interest (FS row 22)
sr_interest = get_row_data(fs, 22)
# Dividends (FS row 24)
dividends_orig = get_row_data(fs, 24)
# Fixed assets closing (Workings row 47)
fa_closing = get_row_data_with_2014(wk, 47)
# Senior debt closing (Workings row 55)
sr_closing = get_row_data_with_2014(wk, 55)
# Net Assets (FS row 60)
net_assets_orig = get_row_data_with_2014(fs, 60)
# Retained earnings (FS row 63)
ret_earnings_orig = get_row_data_with_2014(fs, 63)

wb.close()

print(f"Project CF: {len(project_cf)} years, 2017={project_cf.get(2017):.0f}")
print(f"Sr closing 2016: {sr_closing.get(2016):.0f}")
print(f"FA closing 2020: {fa_closing.get(2020):.2f}")
print(f"Net assets 2020: {net_assets_orig.get(2020):.2f}")

In [ ]:
# PART A: Refinancing Debt Model

def days_in_year(year):
    return 366 if calendar.isleap(year) else 365

drawdown_date = date(2016, 12, 31)
maturity_date = date(2036, 6, 30)
interest_rate = 0.06
arrangement_fee_pct = 0.02

# Existing debt balance after 2016 repayment = closing balance at end 2016
existing_balance = sr_closing[2016]  # 9,000,000

# amount_drawn = existing_balance / (1 - fee_pct)
amount_drawn = existing_balance / (1 - arrangement_fee_pct)
arrangement_fee = amount_drawn * arrangement_fee_pct

print(f"Amount drawn: {amount_drawn:.6f}")
print(f"Arrangement fee: {arrangement_fee:.6f}")

# Build repayment schedule
def days_active_in_year(yr):
    if yr < 2017:
        return 0
    start = date(yr, 1, 1)
    end = date(yr, 12, 31) if yr < 2036 else maturity_date
    if yr == 2036:
        end = maturity_date
        return (end - start).days + 1
    return days_in_year(yr)

refi_schedule = {}
opening = amount_drawn
for yr in range(2017, 2037):
    da = days_active_in_year(yr)
    if da == 0:
        continue
    td = (maturity_date - date(yr, 1, 1)).days + 1
    repayment = opening * da / td
    interest = opening * interest_rate * da / 365
    closing = opening - repayment
    refi_schedule[yr] = {
        'opening': opening, 'repayment': repayment,
        'interest': interest, 'closing': closing,
        'active_days': da, 'total_days': td
    }
    opening = closing

# Arrangement fee amortization
total_active_days = sum(refi_schedule[y]['active_days'] for y in refi_schedule)
amort_schedule = {yr: arrangement_fee * refi_schedule[yr]['active_days'] / total_active_days
                  for yr in refi_schedule}

total_interest = sum(refi_schedule[y]['interest'] for y in refi_schedule)

print(f"\nRepayment 2022: {refi_schedule[2022]['repayment']:.6f}")
print(f"Amortization 2024: {amort_schedule[2024]:.6f}")
print(f"Total interest: {total_interest:.6f}")

In [ ]:
# Part A: Dividends and Net Assets

# Dividends in Part A
# 2015: unchanged from original
# 2016: Project CF - arrangement fee (fee paid from drawdown, but on CF stmt it appears)
#   Actually: drawdown pays fee + repays old debt. Project CF still pays old interest+repayment.
#   Original 2016 dividend = PCF - old_interest - old_repayment = 2040000 - 665000 - 500000 = 875000
#   In Part A, 2016 is unchanged since refi happens at end of 2016 after all other transactions
# 2017+: Project CF - refi_interest - refi_repayment

dividends_a = {}
for yr in range(2015, 2037):
    pcf = project_cf.get(yr, 0)
    if yr <= 2016:
        dividends_a[yr] = abs(dividends_orig.get(yr, 0))
    elif yr in refi_schedule:
        dividends_a[yr] = pcf - refi_schedule[yr]['interest'] - refi_schedule[yr]['repayment']

print(f"Dividend 2034: {dividends_a.get(2034, 0):.6f}")

# Net Assets on 31 Dec 2020 in Part A
# Net Assets = Fixed Assets + Cash + Arr Fee Asset - Refi Debt
# Cash = 0 (all excess distributed)
cum_amort_2020 = sum(amort_schedule[yr] for yr in range(2017, 2021))
arr_fee_asset_2020 = arrangement_fee - cum_amort_2020
net_assets_2020 = fa_closing[2020] + 0 + arr_fee_asset_2020 - refi_schedule[2020]['closing']
print(f"Net Assets 2020: {net_assets_2020:.6f}")

# DSCR calculations (Part A, pre-amendment)
dscrs = []
for yr in sorted(refi_schedule.keys()):
    s = refi_schedule[yr]
    da = s['active_days']
    diy = days_in_year(yr)
    pcf_apportion = project_cf[yr] * da / diy
    debt_service = s['interest'] + s['repayment']
    dscrs.append(pcf_apportion / debt_service)

avg_dscr = sum(dscrs) / len(dscrs)
min_dscr = min(dscrs)
print(f"Average DSCR: {avg_dscr:.6f}")
print(f"Minimum DSCR: {min_dscr:.6f}")

In [ ]:
# PART B: Optimize for DSCR = 1.80
target_dscr = 1.80

def compute_closing_b(drawn):
    balance = drawn
    for yr in range(2017, 2037):
        da = days_active_in_year(yr)
        if da == 0:
            continue
        diy = days_in_year(yr)
        pcf_apportion = project_cf[yr] * da / diy
        target_ds = pcf_apportion / target_dscr
        interest = balance * interest_rate * da / 365
        principal = target_ds - interest
        balance -= principal
    return balance

# Bisection to find optimal drawdown
lo, hi = amount_drawn, amount_drawn * 10
for _ in range(200):
    mid = (lo + hi) / 2
    c = compute_closing_b(mid)
    if abs(c) < 0.0001:
        break
    if c > 0:
        lo = mid
    else:
        hi = mid

draw_b = mid
fee_b = draw_b * arrangement_fee_pct

# Build Part B schedule
balance_b = draw_b
partb_schedule = {}
for yr in range(2017, 2037):
    da = days_active_in_year(yr)
    if da == 0:
        continue
    diy = days_in_year(yr)
    pcf_apportion = project_cf[yr] * da / diy
    target_ds = pcf_apportion / target_dscr
    interest = balance_b * interest_rate * da / 365
    principal = target_ds - interest
    balance_b -= principal
    partb_schedule[yr] = {
        'opening': balance_b + principal, 'interest': interest,
        'principal': principal, 'target_ds': target_ds, 'closing': balance_b
    }

# Q8: target debt service on 30 Jun 2036
ds_2036 = partb_schedule[2036]['target_ds']
print(f"Part B drawdown: {draw_b:.6f}")
print(f"Last 3 digits: {round(draw_b) % 1000}")
print(f"DS 2036: {ds_2036:.6f}")

# Q10: Dividend on 31 Dec 2016 in Part B
# Excess refi proceeds distributed immediately to shareholders
# div = original_2016_dividend + (draw_b - fee_b - existing_balance)
excess = draw_b - fee_b - existing_balance
div_2016_b = abs(dividends_orig[2016]) + excess
print(f"Dividend 2016 Part B: {div_2016_b:.6f}")

In [ ]:
# Delete this cell - merged into cell 3
pass

In [ ]:
# Delete this cell - merged into cell 4
pass

In [ ]:
# Match to multiple choice and set answers
opts = {
    1: {'A': 9183670, 'B': 9183671, 'C': 9183672, 'D': 9183673, 'E': 9183674, 'F': 9183675},
    2: {'A': 470721, 'B': 470722, 'C': 470723, 'D': 470724, 'E': 470725, 'F': 470726},
    3: {'A': 9435, 'B': 9436, 'C': 9437, 'D': 9438, 'E': 9439, 'F': 9440},
    4: {'A': 5647226, 'B': 5647227, 'C': 5647228, 'D': 5647229, 'E': 5647230, 'F': 5647231},
    5: {'A': 2372322, 'B': 2372323, 'C': 2372324, 'D': 2372325, 'E': 2372326, 'F': 2372327},
    6: {'A': 5598333, 'B': 5598334, 'C': 5598335, 'D': 5598336, 'E': 5598337, 'F': 5598338},
    7: {'A': 3.6162, 'B': 3.6212, 'C': 3.6262, 'D': 3.6312, 'E': 3.6362, 'F': 3.6412},
    8: {'A': 836936, 'B': 837036, 'C': 837136, 'D': 837236, 'E': 837336, 'F': 837436},
    9: {'A': 132, 'B': 133, 'C': 134, 'D': 135, 'E': 136, 'F': 137},
    10: {'A': 6818000, 'B': 6819000, 'C': 6820000, 'D': 6821000, 'E': 6822000, 'F': 6823000},
}

vals = {
    1: amount_drawn,
    2: refi_schedule[2022]['repayment'],
    3: amort_schedule[2024],
    4: total_interest,
    5: dividends_a.get(2034, 0),
    6: net_assets_2020,
    7: avg_dscr,
    8: ds_2036,
    9: round(draw_b) % 1000,
    10: div_2016_b,
}

def closest(options, value):
    return min(options, key=lambda k: abs(options[k] - value))

answers = {}
for q in range(1, 11):
    letter = closest(opts[q], vals[q])
    answers[f"question{q}"] = letter
    print(f"Q{q}: {vals[q]:.4f} -> {letter}")

print(f"\nanswers = {answers}")